### Caricamento di un Dataset.

In [4]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("zadafiyabhrami/global-crocodile-species-dataset")

# print("Path to dataset files:", path)

In [5]:
import pandas as pd

In [23]:
df = pd.read_csv(r"data\raw\crocodile_dataset.csv")
df.head()

,Observation ID,Common Name,Scientific Name,Family,Genus,Observed Length (m),Observed Weight (kg),Age Class,Sex,Date of Observation,Country/Region,Habitat Type,Conservation Status,Observer Name,Notes
0,1,Morelet's Crocodile,Crocodylus moreletii,Crocodylidae,Crocodylus,1.90,62.0,Adult,Male,31-03-2018,Belize,Swamps,Least Concern,Allison Hill,Cause bill scientist nation opportunity.
1,2,American Crocodile,Crocodylus acutus,Crocodylidae,Crocodylus,4.09,334.5,Adult,Male,28-01-2015,Venezuela,Mangroves,Vulnerable,Brandon Hall,Ago current practice nation determine operatio...
2,3,Orinoco Crocodile,Crocodylus intermedius,Crocodylidae,Crocodylus,1.08,118.2,Juvenile,Unknown,07-12-2010,Venezuela,Flooded Savannas,Critically Endangered,Melissa Peterson,Democratic shake bill here grow gas enough ana...
3,4,Morelet's Crocodile,Crocodylus moreletii,Crocodylidae,Crocodylus,2.42,90.4,Adult,Male,01-11-2019,Mexico,Rivers,Least Concern,Edward Fuller,Officer relate animal direction eye bag do.
4,5,Mugger Crocodile (Marsh Crocodile),Crocodylus palustris,Crocodylidae,Crocodylus,3.75,269.4,Adult,Unknown,15-07-2019,India,Rivers,Vulnerable,Donald Reid,Class great prove reduce raise author play mov...


In [28]:
import re
import pandas as pd
from typing import Dict, List, Optional

COMMON_DATE_FORMATS = [
    "%Y-%m-%d", "%d-%m-%Y", "%m-%d-%Y",
    "%d/%m/%Y", "%m/%d/%Y", "%d.%m.%Y", "%Y/%m/%d",
    "%d %b %Y", "%d %B %Y",
]
_DATEY_LOOK = re.compile(r"^[\d]{1,4}([\-\/\.\s])[\dA-Za-z]{1,3}\1[\d]{2,4}$")

def _infer_dayfirst(series: pd.Series) -> bool:
    """Euristica: se la prima componente > 12 in molti valori, assumo dayfirst=True."""
    s = series.dropna().astype(str)
    if s.empty: 
        return True  # default EU
    sample = s.sample(min(len(s), 400), random_state=0)
    tokens = sample.str.replace(r"[^\d\/\-\.\s]", "", regex=True).str.split(r"[\/\-\.\s]+", regex=True)
    first_nums = tokens.apply(lambda t: int(t[0]) if t and t[0].isdigit() else None)
    # Conta quante volte il primo numero è >12 (quindi plausibile giorno)
    ratio_day_gt_12 = (first_nums.dropna() > 12).mean() if first_nums.notna().any() else 0.0
    return bool(ratio_day_gt_12 >= 0.2)  # soglia morbida

def try_parse_datetime(series: pd.Series, min_parse_rate: float = 0.8,
                       dayfirst_default: Optional[bool] = None) -> Optional[pd.Series]:
    s = series.dropna()
    if len(s) == 0:
        return None
    # quick pre-check: sembra una data?
    sample = s.astype(str)
    sample = sample.sample(min(len(sample), 200), random_state=0)
    looks_like_date = (
        sample.str.len().between(6, 32) &
        (sample.str.match(_DATEY_LOOK) | sample.str.contains(r"[A-Za-z]{3}", regex=True))
    )
    if looks_like_date.mean() < 0.3:
        return None

    # inferisci dayfirst se non specificato
    dayfirst = _infer_dayfirst(series) if dayfirst_default is None else bool(dayfirst_default)

    # 1) parser moderno (pandas ≥2): format="mixed" per formati misti
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=dayfirst, format="mixed")
    rate = parsed.notna().mean()

    # 2) fallback su formati espliciti
    if rate < min_parse_rate:
        best, best_rate = parsed, rate
        for fmt in COMMON_DATE_FORMATS:
            p = pd.to_datetime(series, errors="coerce", format=fmt)
            r = p.notna().mean()
            if r > best_rate:
                best, best_rate = p, r
            if best_rate >= min_parse_rate:
                break
        parsed, rate = best, best_rate

    return parsed if rate >= min_parse_rate else None

def detect_datetime_columns(df: pd.DataFrame, min_parse_rate: float = 0.8) -> Dict[str, float]:
    results = {}
    for col in df.columns:
        ser = df[col]
        if pd.api.types.is_datetime64_any_dtype(ser):
            results[col] = 1.0
        elif pd.api.types.is_object_dtype(ser) or pd.api.types.is_string_dtype(ser):
            parsed = try_parse_datetime(ser, min_parse_rate=min_parse_rate, dayfirst_default=None)
            if parsed is not None:
                results[col] = float(parsed.notna().mean())
    return results

def coerce_datetime_inplace(df: pd.DataFrame, min_parse_rate: float = 0.8,
                            dayfirst_default: Optional[bool] = None) -> List[str]:
    converted = []
    for col in df.columns:
        ser = df[col]
        if pd.api.types.is_datetime64_any_dtype(ser):
            continue
        if pd.api.types.is_object_dtype(ser) or pd.api.types.is_string_dtype(ser):
            parsed = try_parse_datetime(ser, min_parse_rate=min_parse_rate,
                                        dayfirst_default=dayfirst_default)
            if parsed is not None:
                df[col] = parsed
                converted.append(col)
    return converted


In [29]:
import pandas as pd
import re

_WORD_RE = re.compile(r"\w+", flags=re.UNICODE)

def _text_stats(ser: pd.Series) -> dict:
    s = ser.dropna().astype(str)
    if s.empty:
        return dict(avg_chars=0, avg_words=0, unique_token_ratio=0.0, pct_long_lines=0.0)
    lens = s.str.len()
    words = s.str.findall(_WORD_RE).apply(len)
    sample = s.sample(min(len(s), 500), random_state=0)
    tokens = _WORD_RE.findall(" ".join(sample.tolist()).lower())
    vocab = set(tokens)
    utr = len(vocab) / max(len(tokens), 1)
    return dict(
        avg_chars=float(lens.mean()),
        avg_words=float(words.mean()),
        unique_token_ratio=float(utr),
        pct_long_lines=float((lens >= 120).mean()),
    )

def _looks_like_long_text(stats: dict, min_avg_chars=120, min_avg_words=20, min_utr=0.2):
    return (stats["avg_chars"] >= min_avg_chars or stats["avg_words"] >= min_avg_words) and (stats["unique_token_ratio"] >= min_utr)

def _categorical_values_preview(ser: pd.Series, max_cat_values: int = 10):
    nuniq = ser.nunique(dropna=True)
    if nuniq == 0:
        return [], 0, False
    if nuniq > max_cat_values:
        return None, nuniq, True
    vals = sorted(ser.dropna().astype(str).unique().tolist(), key=lambda x: x.lower())
    return vals, int(nuniq), False

def categorize_columns(
    df: pd.DataFrame,
    threshold_cat: int = 20,
    max_cat_values: int = 10,
    nlp_avg_chars: int = 120,
    nlp_avg_words: int = 20,
    nlp_min_unique_token_ratio: float = 0.2,
    auto_parse_dates: bool = True,
) -> dict:
    if auto_parse_dates:
        coerce_datetime_inplace(df, min_parse_rate=0.8, dayfirst_default=None)

    cats = {
        "numeriche_continue": [],
        "numeriche_discrete": [],
        "categoriche": {},      # {col: {n_unique, values/None, truncated}}
        "boolean": [],
        "datetime": [],
        "testo_libero": [],
        "long_text_nlp": {},    # {col: stats}
    }

    for col in df.columns:
        ser = df[col]

        # boolean
        if pd.api.types.is_bool_dtype(ser) or (pd.api.types.is_integer_dtype(ser) and ser.dropna().isin([0, 1]).all()):
            cats["boolean"].append(col); continue

        # datetime
        if pd.api.types.is_datetime64_any_dtype(ser):
            cats["datetime"].append(col); continue

        # numeric
        if pd.api.types.is_numeric_dtype(ser):
            (cats["numeriche_discrete"] if ser.nunique(dropna=True) <= threshold_cat else cats["numeriche_continue"]).append(col)
            continue

        # text-like
        if pd.api.types.is_object_dtype(ser) or pd.api.types.is_categorical_dtype(ser):
            stats = _text_stats(ser)
            if _looks_like_long_text(stats, nlp_avg_chars, nlp_avg_words, nlp_min_unique_token_ratio):
                cats["long_text_nlp"][col] = stats
            else:
                nuniq = ser.nunique(dropna=True)
                if nuniq <= threshold_cat:
                    values, n_unique, truncated = _categorical_values_preview(ser, max_cat_values=max_cat_values)
                    cats["categoriche"][col] = {"n_unique": n_unique, "values": values, "truncated": truncated}
                else:
                    cats["testo_libero"].append(col)
            continue

        # fallback
        values, n_unique, truncated = _categorical_values_preview(ser, max_cat_values=max_cat_values)
        cats["categoriche"][col] = {"n_unique": n_unique, "values": values, "truncated": truncated}

    return cats


In [35]:
df["Observer Name"].unique()

array(['Allison Hill', 'Brandon Hall', 'Melissa Peterson',
       'Edward Fuller', 'Donald Reid', 'Randy Brown',
       'Dr. Marvin Thomas Jr.', 'Terri Frazier', 'Deborah Mason',
       'Tamara George', 'Betty Alvarez', 'Jennifer Powers', 'Mark Perez',
       'Timothy Duncan', 'Matthew Lucas', 'Donald Wright', 'Sarah Martin',
       'Amy Edwards', 'Kurt Leonard', 'Matthew Cunningham', 'Paul Jones',
       'Dustin Kim', 'Mark Baker', 'Jacob Tran', 'David Garcia',
       'Shane Garrison', 'Donald Jones', 'Mrs. Linda Reed',
       'Christina Reynolds', 'Jamie Adkins', 'Julie Ramos', 'Eric Drake',
       'Matthew Smith', 'Casey Marshall', 'Denise Jones', 'Ross Price',
       'Morgan Marsh', 'Mary Miller', 'Paul Decker', 'Lisa Allen',
       'Miguel Jones', 'Christopher Boyd', 'Laura Barnett',
       'Peter Vaughn DDS', 'Eric Hall', 'Paige Carlson', 'Edgar Raymond',
       'Tiffany Miller', 'Elizabeth Oliver DDS', 'Jerry Wheeler',
       'Dennis Moody', 'Melissa Bender', 'Joseph Ramos', 'Ga

In [34]:
categorize_columns(df, threshold_cat=50)

{'numeriche_continue': ['Observation ID',
  'Observed Length (m)',
  'Observed Weight (kg)'],
 'numeriche_discrete': [],
 'categoriche': {'Common Name': {'n_unique': 18,
   'values': None,
   'truncated': True},
  'Scientific Name': {'n_unique': 18, 'values': None, 'truncated': True},
  'Family': {'n_unique': 1, 'values': ['Crocodylidae'], 'truncated': False},
  'Genus': {'n_unique': 3,
   'values': ['Crocodylus', 'Mecistops', 'Osteolaemus'],
   'truncated': False},
  'Age Class': {'n_unique': 4,
   'values': ['Adult', 'Hatchling', 'Juvenile', 'Subadult'],
   'truncated': False},
  'Sex': {'n_unique': 3,
   'values': ['Female', 'Male', 'Unknown'],
   'truncated': False},
  'Country/Region': {'n_unique': 47, 'values': None, 'truncated': True},
  'Habitat Type': {'n_unique': 29, 'values': None, 'truncated': True},
  'Conservation Status': {'n_unique': 5,
   'values': ['Critically Endangered',
    'Data Deficient',
    'Endangered',
    'Least Concern',
    'Vulnerable'],
   'truncated': 